In [ ]:
!pip install -q transformers peft torch safetensors accelerate

In [ ]:
import torch
print(torch.cuda.is_available())        # should print True
print(torch.cuda.get_device_name(0))    # should print Tesla T4

True
Tesla T4


In [ ]:
!unzip /content/rykyu_adapter.zip -d /content/

Archive:  /content/rykyu_adapter.zip
   creating: /content/rykyu_adapter/
  inflating: /content/rykyu_adapter/adapter_model.safetensors  
  inflating: /content/rykyu_adapter/INSTRUCTIONS.md  
  inflating: /content/rykyu_adapter/tokenizer_config.json  
  inflating: /content/rykyu_adapter/adapter_config.json  
  inflating: /content/rykyu_adapter/merge_and_export.py  
  inflating: /content/rykyu_adapter/Modelfile  
  inflating: /content/rykyu_adapter/chat_template.jinja  


In [ ]:
ADAPTER_PATH = "/content/rykyu_adapter"
BASE_MODEL   = "unsloth/Llama-3.2-3B-Instruct"
OUTPUT_PATH  = "/content/merged_model"
GGUF_OUT     = "/content/rykyu-math-tutor.gguf"

In [ ]:
!pip install -q torchao --upgrade
import importlib, peft
importlib.reload(peft)
from peft import PeftModel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.4 MB/s eta 0:00:00


In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading base model...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Applying LoRA adapter...")
model = PeftModel.from_pretrained(base, ADAPTER_PATH)

print("Merging weights...")
merged = model.merge_and_unload()

print("Saving merged model...")
merged.save_pretrained(OUTPUT_PATH, safe_serialization=True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.save_pretrained(OUTPUT_PATH)
print("✓ Merged model saved to", OUTPUT_PATH)

Loading base model...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Applying LoRA adapter...
Merging weights...
Saving merged model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

✓ Merged model saved to /content/merged_model


In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt

!python /content/llama.cpp/convert_hf_to_gguf.py \
    {OUTPUT_PATH} \
    --outfile {GGUF_OUT} \
    --outtype q4_k_m

print("✓ GGUF saved to", GGUF_OUT)

Cloning into '/content/llama.cpp'...
remote: Enumerating objects: 3320, done.
remote: Counting objects: 100% (3320/3320), done.
remote: Compressing objects: 100% (2644/2644), done.
remote: Total 3320 (delta 627), reused 2958 (delta 597), pack-reused 0 (from 0)
Receiving objects: 100% (3320/3320), 33.36 MiB | 14.99 MiB/s, done.
Resolving deltas: 100% (627/627), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 71.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 113.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
size = os.path.getsize(GGUF_OUT) / (1024**3)
print(f"GGUF file size: {size:.2f} GB")
print("Done! Download the .gguf from Drive and run:")
print("  ollama create rykyu-math-tutor -f Modelfile")
print("  ollama run rykyu-math-tutor")

FileNotFoundError: [Errno 2] No such file or directory: '/content/rykyu-math-tutor.gguf'

In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/merged_model \
    --outfile /content/rykyu-math-tutor.gguf \
    --outtype q4_k_m

usage: convert_hf_to_gguf.py [-h] [--vocab-only] [--outfile OUTFILE]
                             [--outtype {f32,f16,bf16,q8_0,tq1_0,tq2_0,auto}]
                             [--bigendian] [--use-temp-file] [--no-lazy]
                             [--model-name MODEL_NAME] [--verbose]
                             [--split-max-tensors SPLIT_MAX_TENSORS]
                             [--split-max-size SPLIT_MAX_SIZE] [--dry-run]
                             [--no-tensor-first-split] [--metadata METADATA]
                             [--print-supported-models] [--remote] [--mmproj]
                             [--mtp] [--no-mtp] [--mistral-format]
                             [--disable-mistral-community-chat-template]
                             [--sentence-transformers-dense-modules]
                             [--fuse-gate-up-exps] [--fp8-as-q8]
                             [--target-model-dir TARGET_MODEL_DIR]
                             [model]
convert_hf_to_gguf.py: error: argume

In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/merged_model \
    --outfile /content/rykyu-math-tutor-f16.gguf \
    --outtype f16

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {3072, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {8192, 3072}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {3072, 1024}
INFO:hf-to-gguf:blk.0.attn_output.weight,  

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-3B-Instruct", use_fast=False)
tokenizer.save_pretrained("/content/merged_model")

('/content/merged_model/tokenizer_config.json',
 '/content/merged_model/chat_template.jinja',
 '/content/merged_model/tokenizer.json')

In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/merged_model \
    --outfile /content/rykyu-math-tutor-f16.gguf \
    --outtype f16

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {3072, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {8192, 3072}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {3072, 1024}
INFO:hf-to-gguf:blk.0.attn_output.weight,  

In [ ]:
import json, os

# Fix the tokenizer_config.json - replace TokenizersBackend with PreTrainedTokenizerFast
tokenizer_config = {
  "bos_token": "<|begin_of_text|>",
  "eos_token": "<|eot_id|>",
  "pad_token": "<|eot_id|>",
  "model_max_length": 131072,
  "tokenizer_class": "PreTrainedTokenizerFast",
  "clean_up_tokenization_spaces": True
}

with open("/content/merged_model/tokenizer_config.json", "w") as f:
    json.dump(tokenizer_config, f, indent=2)

# Verify what tokenizer files exist now
print(os.listdir("/content/merged_model"))

['tokenizer.json', 'chat_template.jinja', 'tokenizer_config.json', 'generation_config.json', 'config.json', 'model.safetensors']


In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/merged_model \
    --outfile /content/rykyu-math-tutor-f16.gguf \
    --outtype f16

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {3072, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {8192, 3072}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {3072, 1024}
INFO:hf-to-gguf:blk.0.attn_output.weight,  

In [ ]:
!cd /content/llama.cpp && make llama-quantize -j4

!/content/llama.cpp/llama-quantize \
    /content/rykyu-math-tutor-f16.gguf \
    /content/rykyu-math-tutor.gguf \
    Q4_K_M

Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.
/bin/bash: line 1: /content/llama.cpp/llama-quantize: No such file or directory


In [ ]:
!cd /content/llama.cpp && cmake -B build && cmake --build build --config Release -j4 --target llama-quantize

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found OpenMP_C: -fopenmp (found version "

In [ ]:
!/content/llama.cpp/build/bin/llama-quantize \
    /content/rykyu-math-tutor-f16.gguf \
    /content/rykyu-math-tutor.gguf \
    Q4_K_M

llama_print_build_info: build = 1 (bae36ef)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/content/rykyu-math-tutor-f16.gguf' to '/content/rykyu-math-tutor.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 31 key-value pairs and 255 tensors from /content/rykyu-math-tutor-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_p f32              = 0.900000
llama_model_loader: - kv   3:                      general.sampling.temp f32              = 0.600000
llama_model_loader: - kv   4:                               general.name str              = Merged_Model
llama_model_loader: - kv   5:         

In [ ]:
from google.colab import files
files.download('/content/rykyu-math-tutor.gguf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Run this in Colab to check the file is valid
!head -c 4 /content/rykyu-math-tutor.gguf | xxd

00000000: 4747 5546                                GGUF


In [ ]:
!zip /content/rykyu-math-tutor.zip /content/rykyu-math-tutor.gguf
from google.colab import files
files.download('/content/rykyu-math-tutor.zip')


  adding: content/rykyu-math-tutor.gguf (deflated 2%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>